# SleepSync – Sleep & Mood Relationship Analyzer

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
demographic = pd.read_csv("dataset/demographic.csv")
phq = pd.read_csv("dataset/phq9.csv")
gad = pd.read_csv("dataset/gad7.csv")
isi = pd.read_csv("dataset/isi.csv")
pss = pd.read_csv("dataset/pss.csv")

In [ ]:
print("Demographic shape:", demographic.shape)
print("PHQ-9 shape:", phq.shape)
print("GAD-7 shape:", gad.shape)
print("ISI shape:", isi.shape)
print("PSS shape:", pss.shape)

In [ ]:
print(demographic.head())

In [ ]:
print(phq.head())

In [ ]:
print("PHQ-9 columns:", phq.columns.tolist())

In [ ]:
phq_items = phq[["export_id"] + ["question" + str(i) for i in range(1, 10)]].copy()
phq_items.columns = ["export_id"] + ["phq_" + str(i) for i in range(1, 10)]
phq_items["phq_total"] = phq["score"]

In [ ]:
gad_items = gad[["export_id"] + ["question" + str(i) for i in range(1, 8)]].copy()
gad_items.columns = ["export_id"] + ["gad_" + str(i) for i in range(1, 8)]
gad_items["gad_total"] = gad["score"]

In [ ]:
isi_items = isi[["export_id"] + ["question" + str(i) for i in range(1, 8)]].copy()
isi_items.columns = ["export_id"] + ["isi_" + str(i) for i in range(1, 8)]
isi_items["isi_total"] = isi["score"]

In [ ]:
pss_total = pss[["export_id", "score"]].copy()
pss_total.columns = ["export_id", "pss_total"]

In [ ]:
df = demographic.merge(phq_items, on="export_id")
print("After PHQ-9 merge:", df.shape)

In [ ]:
df = df.merge(gad_items, on="export_id")
print("After GAD-7 merge:", df.shape)

In [ ]:
df = df.merge(isi_items, on="export_id")
print("After ISI merge:", df.shape)

In [ ]:
df = df.merge(pss_total, on="export_id")
print("After PSS merge:", df.shape)

In [ ]:
print("Merged dataset shape:", df.shape)
print(df.columns.tolist())

In [ ]:
df.head()

In [ ]:
print("Dataset shape:", df.shape)
print(df.head())

In [ ]:
print(df.info())

In [ ]:
df.describe()

In [ ]:
print(df.isnull().sum().sum())

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate student ids:", df["export_id"].duplicated().sum())

In [ ]:
for col in ["gender", "edu", "smoke", "drink"]:
    print("\n", col)
    print(df[col].value_counts())

In [ ]:
print("Minimum age:", df["age"].min())
print("Maximum age:", df["age"].max())
print("Students with age below 16:", (df["age"] < 16).sum())
print("Students with age above 40:", (df["age"] > 40).sum())

In [ ]:
phq_cols = ["phq_" + str(i) for i in range(1, 10)]
gad_cols = ["gad_" + str(i) for i in range(1, 8)]
isi_cols = ["isi_" + str(i) for i in range(1, 8)]

In [ ]:
print("PHQ-9 total matches item sum:", (df[phq_cols].sum(axis=1) == df["phq_total"]).mean())
print("GAD-7 total matches item sum:", (df[gad_cols].sum(axis=1) == df["gad_total"]).mean())
print("ISI total matches item sum:", (df[isi_cols].sum(axis=1) == df["isi_total"]).mean())

In [ ]:
print("PHQ-9 item range:", df[phq_cols].min().min(), "to", df[phq_cols].max().max())
print("GAD-7 item range:", df[gad_cols].min().min(), "to", df[gad_cols].max().max())
print("ISI item range:", df[isi_cols].min().min(), "to", df[isi_cols].max().max())

In [ ]:
print("PHQ-9 total range:", df["phq_total"].min(), "to", df["phq_total"].max(), "(possible 0 to 27)")
print("GAD-7 total range:", df["gad_total"].min(), "to", df["gad_total"].max(), "(possible 0 to 21)")
print("ISI total range:", df["isi_total"].min(), "to", df["isi_total"].max(), "(possible 0 to 28)")
print("PSS total range:", df["pss_total"].min(), "to", df["pss_total"].max(), "(possible 0 to 56)")

In [ ]:
print("Correlation between PHQ-9 question 3 and ISI total:", round(df["phq_3"].corr(df["isi_total"]), 2))

In [ ]:
print("Dataset shape:", df.shape)

In [ ]:
df.columns = df.columns.str.strip()

In [ ]:
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip().str.lower()

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate student ids:", df["export_id"].duplicated().sum())

In [ ]:
df = df.drop_duplicates()
df = df.drop_duplicates(subset="export_id")

In [ ]:
print("Dataset shape after removing duplicates:", df.shape)

In [ ]:
print(df.isnull().sum().sum())

In [ ]:
num_cols = ["age", "phq_total", "gad_total", "isi_total", "pss_total"]
for col in num_cols:
    plt.figure(figsize=(8, 3))
    sns.boxplot(x=df[col])
    plt.title("Outlier Check: " + col)
    plt.show()

In [ ]:
outlier_results = []
for col in num_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_results.append({
        "Column": col,
        "Lower Bound": round(lower, 2),
        "Upper Bound": round(upper, 2),
        "Flagged Values": int(count)
    })
outlier_df = pd.DataFrame(outlier_results)
print(outlier_df.to_string(index=False))

In [ ]:
print("Minimum age:", df["age"].min())
print("Maximum age:", df["age"].max())

In [ ]:
invalid_age = (df["age"] < 16) | (df["age"] > 40)
print("Rows with invalid age:", invalid_age.sum())

In [ ]:
df = df[~invalid_age].copy()
print("Dataset shape after age filter:", df.shape)

In [ ]:
print("Minimum age:", df["age"].min())
print("Maximum age:", df["age"].max())

In [ ]:
phq_no_sleep = ["phq_" + str(i) for i in range(1, 10) if i != 3]
print(phq_no_sleep)

In [ ]:
df["mood_score"] = df[phq_no_sleep].sum(axis=1)
print(df["mood_score"].describe())

In [ ]:
df["anxiety_score"] = df["gad_total"]

In [ ]:
df["female"] = (df["gender"] == "female").astype(int)

The target low_mood is 1 when the mood score is 5 or more (mild or higher mood symptoms on the PHQ-9 scale). The usual PHQ-9 cutoff of 10 would flag only about 4% of students, which is too few for a model to learn from.

In [ ]:
for t in [5, 8, 10]:
    print("Mood score >=", t, ":", round((df["mood_score"] >= t).mean() * 100, 2), "% of students")

In [ ]:
df["low_mood"] = (df["mood_score"] >= 5).astype(int)
print(df["low_mood"].value_counts())
print(df["low_mood"].value_counts(normalize=True).round(3))

In [ ]:
df["sleep_group"] = pd.cut(
    df["isi_total"],
    bins=[-1, 7, 14, 28],
    labels=["No problem", "Mild", "Moderate or severe"]
)
print(df["sleep_group"].value_counts())

In [ ]:
print("Dataset shape:", df.shape)
print(df.isnull().sum().sum())

In [ ]:
df.to_csv("cleaned_data.csv", index=False)
print("cleaned_data.csv saved")

In [ ]:
print("Dataset shape:", df.shape)

In [ ]:
print(df["low_mood"].value_counts(normalize=True).round(3))
plt.figure(figsize=(5, 4))
sns.countplot(x="low_mood", data=df)
plt.title("Low Mood Distribution")
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df["isi_total"], bins=29)
plt.title("Insomnia Severity Score (ISI)")
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df["mood_score"], bins=25)
plt.title("Mood Symptom Score (PHQ-9 without sleep question)")
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x="gender", data=df)
plt.title("Gender")
plt.show()

In [ ]:
sleep_summary = df.groupby("sleep_group", observed=True)[["mood_score", "anxiety_score", "pss_total"]].mean().round(2)
sleep_summary["students"] = df["sleep_group"].value_counts()
print(sleep_summary)

In [ ]:
sleep_summary[["mood_score", "anxiety_score"]].plot(kind="bar", figsize=(7, 4))
plt.title("Sleep Problem Level vs Mood and Anxiety")
plt.ylabel("Average score")
plt.xticks(rotation=0)
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
sns.boxplot(x="sleep_group", y="mood_score", data=df)
plt.title("Mood Score by Sleep Problem Level")
plt.show()

In [ ]:
print(pd.crosstab(df["sleep_group"], df["low_mood"], normalize="index").round(3))

In [ ]:
for col in ["gender", "smoke", "drink"]:
    print("\n", col)
    print(pd.crosstab(df[col], df["low_mood"], normalize="index").round(3))

In [ ]:
corr_cols = ["isi_total", "mood_score", "anxiety_score", "pss_total", "age"]
plt.figure(figsize=(7, 5))
sns.heatmap(df[corr_cols].corr(method="spearman"), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Spearman Correlation")
plt.show()

In [ ]:
isi_cols = ["isi_" + str(i) for i in range(1, 8)]
print(df[isi_cols + ["mood_score"]].corr(method="spearman")["mood_score"].drop("mood_score").round(2))